# Explainable Rice Supply Forecasting Using LSTM and Internal RAG
### A Case Study of Karawang Rice Supply (2011–2025)

**Framework:**

```
Rice Supply Dataset
        ↓
Seasonality Analysis
        ↓
LSTM Forecasting
        ↓
Knowledge Extraction
        ↓
Sentence Transformer
        ↓
ChromaDB
        ↓
Retriever
        ↓
Gemini
        ↓
Explainable Forecast
        ↓
RAGAS Evaluation
```

This notebook is fully self-contained — the dataset is embedded directly (no external files needed) and the only external dependency at runtime is a free Google Gemini API key.

**Model (Model A):** LSTM + Internal RAG, no external variables at this stage. Internal RAG means the knowledge base is built from the dataset itself (monthly averages, yearly trend, seasonal extremes, descriptive stats) rather than external documents — also called Data-to-Text RAG.

Run the cells below in order.

## 1. Install dependencies

In [ ]:
!pip install -q chromadb sentence-transformers google-generativeai langchain-google-genai ragas datasets statsmodels

# chromadb needs sqlite3 >= 3.35; Colab's bundled sqlite3 is often older.
# This swaps in pysqlite3-binary's newer sqlite3 before chromadb is imported.
!pip install -q pysqlite3-binary
import sys
try:
    __import__("pysqlite3")
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
    print("✓ Patched sqlite3 with pysqlite3-binary")
except ImportError:
    print("! pysqlite3 not available, using system sqlite3 (may fail if too old)")

## 2. Get a free Gemini API key

1. Go to https://aistudio.google.com/apikey
2. Click "Create API key" and copy it.
3. Run the next cell **on its own** (not via "Run all") and wait for the input box to appear before pasting. The cell after it does a live test call so you'll know immediately if the key works.

In [ ]:
import getpass

while True:
    GOOGLE_API_KEY = getpass.getpass("Enter your Google AI Studio API key: ").strip()
    if not GOOGLE_API_KEY:
        print("\u274c Empty input \u2014 the key box may not have been ready. Try again.")
        continue
    break

print(f"\u2713 Key captured (length: {len(GOOGLE_API_KEY)}). Run the next cell to verify it works.")

In [ ]:
# Sanity-check the key with a live call before building anything else
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)
GEMINI_MODEL_NAME = "gemini-2.5-flash"

try:
    _model = genai.GenerativeModel(GEMINI_MODEL_NAME)
    _resp = _model.generate_content("Say OK")
    print("\u2713 Key works:", _resp.text.strip())
except Exception as e:
    print("\u274c Key test failed:", e)
    print("Re-run the previous cell and paste the key again.")

## 3. Load Dataset

The dataset (`ricesupply_2011-2025.csv`, monthly rice supply by region, Jan 2011–Apr 2025) is embedded below.

In [ ]:
%%writefile ricesupply_2011-2025.csv
,Tahun,Bulan,Karawang,Cirebon,Bandung,Cianjur,Banten,Jateng,Jatim,Gdg.Jkt,Bulog,Anpu,Total Pasokan
1,2011,2011-01,11114,14991,7175,543,229,16269,4279,2427,22030,177,79234
2,2011,2011-02,13527,18099,7228,519,767,14965,2745,1617,5445,112,65024
3,2011,2011-03,22420,26227,9337,563,590,9145,907,960,5190,193,75532
4,2011,2011-04,23075,26939,8264,587,251,5152,446,280,25,78,65097
5,2011,2011-05,23708,27555,8911,674,474,6707,749,802,0,55,69635
6,2011,2011-06,21536,21858,9243,549,739,6753,681,3715,110,78,65262
7,2011,2011-07,18658,22739,9255,511,495,7239,672,6418,5379,375,71741
8,2011,2011-08,16009,20520,6287,408,222,3572,55,3280,4140,315,54808
9,2011,2011-09,22974,26754,8377,516,216,2503,122,1134,5790,826,69212
10,2011,2011-10,20549,24516,6901,426,54,2412,147,2885,7345,127,65362
11,2011,2011-11,19737,27071,7098,531,8,3951,68,3702,25115,645,87926
12,2011,2011-12,14922,21156,6615,429,37,4399,687,2434,35245,1179,87103
1,2012,2012-01,10241,14485,4882,288,10,11844,3863,1603,27865,1275,76356
2,2012,2012-02,10968,12567,4273,267,100,19736,3016,3502,31095,4072,89596
3,2012,2012-03,20883,17589,7302,630,326,15400,2223,6298,4620,2472,77743
4,2012,2012-04,22042,23187,9621,510,199,8998,1637,3816,1420,595,72025
5,2012,2012-05,19068,25573,9402,302,172,8103,1570,1938,208,714,67050
6,2012,2012-06,12720,18805,7604,304,90,16388,4175,8546,1599,791,71022
7,2012,2012-07,14645,19139,8005,333,27,20176,4296,5655,1100,779,74155
8,2012,2012-08,13532,15642,5483,226,134,9666,1380,2151,0,108,48322
9,2012,2012-09,18281,22940,7331,348,99,16401,2060,3244,140,342,71186
10,2012,2012-10,17277,21524,7410,310,9,14280,1039,2114,240,626,64829
11,2012,2012-11,16294,22177,7436,357,22,17120,3239,1025,1175,396,69241
12,2012,2012-12,11526,15986,6290,260,22,13300,5717,1049,11703,158,66011
1,2013,2013-01,9723,14606,6569,359,0,15446,6593,1560,13833,106,68795
2,2013,2013-02,7114,10383,5721,354,90,18959,3279,5730,3800,219,55649
3,2013,2013-03,13470,17416,7460,402,567,18724,4621,7603,910,378,71551
4,2013,2013-04,17046,23242,10364,518,324,12929,2928,6933,102,396,74782
5,2013,2013-05,18219,22237,10282,482,72,11006,1792,4675,80,306,69151
6,2013,2013-06,13211,19262,8327,476,55,12283,1916,5926,400,823,62679
7,2013,2013-07,12948,18092,8345,479,27,17431,2108,3606,3960,884,67880
8,2013,2013-08,13064,17449,6073,280,63,7513,1099,2545,0,456,48542
9,2013,2013-09,17382,23699,7669,449,18,9993,614,10317,326,225,70692
10,2013,2013-10,18353,25200,7616,469,34,10146,664,9133,536,383,72534
11,2013,2013-11,16930,22962,7389,450,0,9168,580,5861,590,480,64410
12,2013,2013-12,12369,17723,6734,416,27,9342,862,4049,2690,310,54522
1,2014,2014-01,11434,16536,6065,394,36,11291,1007,2075,6065,258,55161
2,2014,2014-02,6225,10978,5094,350,68,15174,1882,1380,11240,3730,56121
3,2014,2014-03,7439,12354,5424,455,180,17868,1853,3005,6650,5681,60909
4,2014,2014-04,14203,19812,7405,508,0,14777,2192,2276,0,2473,63646
5,2014,2014-05,18281,20924,9038,524,0,13020,2676,1667,60,2291,68481
6,2014,2014-06,13847,17611,7534,468,18,18106,2597,2098,250,2930,65459
7,2014,2014-07,10366,13630,5350,360,0,19482,3030,1715,400,1457,55790
8,2014,2014-08,20933,25190,8278,512,0,16988,3805,3979,240,2586,82511
9,2014,2014-09,19265,24325,7897,510,0,12742,2861,10374,40,1502,79516
10,2014,2014-10,20614,24643,7677,504,0,13468,1465,13724,1282,828,84205
11,2014,2014-11,15742,19430,6881,436,0,11798,1730,3560,7140,1287,68004
12,2014,2014-12,14132,17335,6585,458,36,11506,2029,5274,22510,1232,81097
1,2015,2015-01,9086,12163,5116,391,0,20496,2874,4179,30620,1096,86021
2,2015,2015-02,4512,6170,3406,234,0,21505,2477,3610,3904,3887,49705
3,2015,2015-03,12193,15342,5621,519,0,25367,3123,4679,29830,4100,100774
4,2015,2015-04,22113,24457,8157,612,0,25753,3317,2011,150,1803,88373
5,2015,2015-05,23338,25672,8206,475,0,15554,2533,1748,120,934,78580
6,2015,2015-06,23491,27051,7957,440,0,18573,3182,2553,0,864,84111
7,2015,2015-07,14306,16846,5280,326,0,12761,2004,1886,36,389,53834
8,2015,2015-08,20713,25588,7866,536,0,21833,4238,3772,18,1242,85806
9,2015,2015-09,17639,21895,7277,427,0,16353,3235,4761,290,414,72291
10,2015,2015-10,19532,24915,8080,500,0,22455,3221,4439,63,441,83646
11,2015,2015-11,26340,25046,7157,397,36,17867,3288,1357,0,170,81658
12,2015,2015-12,21552,23400,5736,404,0,12839,2038,2073,24840,380,93262
1,2016,2016-01,15711,20473,5121,406,0,21629,4147,1219,31560,449,100715
2,2016,2016-02,13567,16189,4896,446,35,20978,2313,1597,35515,446,95982
3,2016,2016-03,15957,16469,5407,571,330,21891,2709,840,2870,1817,68861
4,2016,2016-04,24580,20661,7735,931,481,17753,2455,747,332,3813,35251
5,2016,2016-05,25318,20788,8253,1284,339,18381,3050,2663,9694,4422,94192
6,2016,2016-06,20909,19579,6592,674,427,20863,3264,3357,40513,2916,119094
7,2016,2016-07,14019,12505,4105,843,417,14729,2089,1059,15897,619,66282
8,2016,2016-08,19182,23674,6914,1116,497,19776,2745,4868,27361,962,107095
9,2016,2016-09,19951,21476,6871,1007,426,17414,1790,2367,11453,1600,84355
10,2016,2016-10,16112,22797,7185,668,696,20135,2924,2371,3434,1035,77357
11,2016,2016-11,21604,22302,7173,603,522,17603,3058,2665,394,1075,76999
12,2016,2016-12,20129,21943,6555,1274,981,12700,3233,3451,8018,762,79046
1,2017,2017-01,18285,25310,6450,818,544,14478,2368,1447,5797,1331,76828
2,2017,2017-02,15467,19750,5913,535,449,19515,3074,645,5018,1592,71958
3,2017,2017-03,15826,22475,7811,982,542,19401,2335,3944,6220,1591,81127
4,2017,2017-04,16485,16581,6088,660,241,17217,2946,7473,3699,1380,72770
5,2017,2017-05,18814,19186,7427,704,417,22163,2719,8790,2826,1212,84258
6,2017,2017-06,14341,11190,3758,687,278,14360,2064,2795,118,925,50516
7,2017,2017-07,20205,17765,6543,1153,436,21077,3240,2324,212,961,73916
8,2017,2017-08,17663,22642,5983,770,330,21813,3364,2687,18,771,76041
9,2017,2017-09,12698,15060,4969,704,260,21275,3620,12089,396,4599,75670
10,2017,2017-10,15798,15046,5676,461,572,22049,3188,4333,4838,11878,83839
11,2017,2017-11,18555,16116,4222,1130,277,19566,2678,7655,9620,1850,81669
12,2017,2017-12,14044,10244,4369,639,255,13750,2728,11872,14549,6557,79007
1,2018,2018-01,8730,5620,2990,502,157,16319,5196,1558,28864,15625,85560
2,2018,2018-02,7938,9065,2591,622,200,22227,4538,2634,21508,18911,90234
3,2018,2018-03,16131,20735,5438,486,269,24897,3286,4848,648,18780,95518
4,2018,2018-04,22016,18102,6202,1292,225,16649,3097,2155,0,12804,82541
5,2018,2018-05,25677,22722,7069,567,368,15855,3923,5597,0,10943,92721
6,2018,2018-06,15023,14992,3186,480,107,12234,1598,1172,0,4492,53284
7,2018,2018-07,23475,34803,8293,463,297,22905,4540,1007,0,3768,99550
8,2018,2018-08,16337,22211,5146,475,103,21644,5872,578,0,5049,77414
9,2018,2018-09,15588,19987,5406,323,323,17239,3517,142,1145,11488,75157
10,2018,2018-10,18784,22558,6470,874,105,21222,3366,350,595,4572,78896
11,2018,2018-11,16238,23381,5566,587,203,16458,1923,2,1628,6984,72970
12,2018,2018-12,14060,17980,4397,710,85,13072,2344,64,10453,5805,68969
1,2019,2019-01,12693,15854,4839,501,289,22850,5415,0,12566,8400,83407
2,2019,2019-02,9175,8994,3222,376,84,24630,2181,85,9303,4382,62432
3,2019,2019-03,13128,14759,4848,681,221,25633,3763,36,2248,7095,72411
4,2019,2019-04,21850,22067,6898,368,352,21737,4184,18,247,6240,83960
5,2019,2019-05,28196,25446,7282,608,922,19731,3090,8,40,5398,90720
6,2019,2019-06,19582,14084,3860,564,184,12050,1763,0,0,1997,54084
7,2019,2019-07,19689,25055,6186,493,871,23279,2811,48,0,2059,80490
8,2019,2019-08,14582,20715,4755,412,246,23713,3355,0,0,3491,71269
9,2019,2019-09,14665,19181,4815,288,340,21566,2498,0,58,7901,71312
10,2019,2019-10,18329,21204,4941,453,235,21777,2868,0,2648,2896,75351
11,2019,2019-11,19180,18256,4674,283,821,16883,2029,0,2210,5649,69985
12,2019,2019-12,18692,21961,4085,319,350,14457,1554,0,2423,4413,68254
1,2020,2020-01,14221,20625,3879,381,646,19133,2187,0,3504,6141,70716
2,2020,2020-02,12603,13364,3083,449,413,19370,2598,0,2313,5310,59503
3,2020,2020-03,13169,15759,3779,474,458,34735,4391,489,3221,9425,85898
4,2020,2020-04,18032,23352,5042,345,579,30445,3728,54,2822,6778,91176
5,2020,2020-05,16439,16826,3874,497,548,9993,1661,419,1396,7014,58666
6,2020,2020-06,25575,21218,5417,474,163,10810,4078,291,1932,2965,72923
7,2020,2020-07,22522,18476,3976,744,507,10041,2716,37,1571,2118,62708
8,2020,2020-08,19057,26879,5556,552,592,12577,3037,85,822,1395,70550
9,2020,2020-09,17874,27953,6470,568,708,12213,2235,0,2108,2610,72738
10,2020,2020-10,20138,24233,5893,382,1246,12767,2336,0,1849,3809,72651
11,2020,2020-11,27377,23559,5886,413,645,11263,1773,0,2352,2177,75443
12,2020,2020-12,29212,26350,5476,677,635,12748,2104,10,2261,3119,82590
1,2021,2021-01,19374,22325,4486,591,615,10018,2887,0,1344,2101,63741
2,2021,2021-02,13297,16934,4113,323,490,19608,2787,0,280,3773,61605
3,2021,2021-03,18697,22229,5484,451,479,25432,2984,0,1269,4887,81912
4,2021,2021-04,22353,21082,6805,489,875,18598,2679,0,1078,8686,82645
5,2021,2021-05,19270,18923,4320,374,378,11347,1617,0,175,5537,61941
6,2021,2021-06,19731,22557,5390,990,832,16212,3898,738,241,5241,75830
7,2021,2021-07,23651,22979,5778,532,368,20687,3142,325,350,2386,80198
8,2021,2021-08,21497,22253,6052,617,516,17149,1708,875,526,2556,73749
9,2021,2021-09,17196,22298,5886,280,229,18980,2189,652,10,3301,71021
10,2021,2021-10,15544,18361,5373,239,280,17284,2077,3170,530,4199,67057
11,2021,2021-11,18378,21973,6105,353,293,19104,3894,665,140,3984,74889
12,2021,2021-12,19193,15877,5382,246,146,21856,4030,225,518,3164,70637
1,2022,2022-01,16164,13791,4111,429,389,20792,3035,0,447,3542,62700
2,2022,2022-02,12603,12128,3186,290,228,22474,2607,0,130,2423,56069
3,2022,2022-03,20583,20437,6434,406,336,30219,3534,25,68,4452,86494
4,2022,2022-04,23870,26031,7159,674,446,18836,2009,9,0,5379,84413
5,2022,2022-05,16549,20150,5133,295,641,11820,1304,796,50,5746,62484
6,2022,2022-06,23839,27407,6856,561,229,20189,3550,1395,745,6294,91065
7,2022,2022-07,44310,39520,14486,824,3098,46154,7318,0,120,8418,164248
8,2022,2022-08,20864,20370,5148,426,436,18930,2734,845,1542,5629,76924
9,2022,2022-09,22485,29929,6249,297,438,17349,3093,150,5153,6569,91712
10,2022,2022-10,23332,25262,5775,214,689,11077,2142,525,3315,4718,77049
11,2022,2022-11,28140,17969,6231,350,649,13909,1617,570,8517,4217,82169
12,2022,2022-12,24302,14246,4564,232,519,15476,2226,0,2120,3504,67189
1,2023,2023-01,18481,5816,2976,119,533,16538,3004,495,8647,7959,64568
2,2023,2023-02,9010,3947,2287,196,1324,14476,1264,0,40623,6146,79273
3,2023,2023-03,17376,14223,4389,402,359,22898,2061,0,18298,8971,88977
4,2023,2023-04,17104,13847,4112,374,395,14858,1157,0,0,2846,54693
5,2023,2023-05,21962,24487,5543,610,1159,15116,1828,70,98,4695,75568
6,2023,2023-06,15133,13262,4568,234,1520,19452,2694,111,270,3018,60262
7,2023,2023-07,15790,17382,4558,294,455,17723,2576,0,36,2975,61789
8,2023,2023-08,15521,14253,4819,317,223,27201,3441,390,185,2587,68937
9,2023,2023-09,13346,13662,3702,243,304,15511,1858,325,5836,6098,60885
10,2023,2023-10,16018,20621,3908,284,233,9953,696,440,16590,1326,70069
11,2023,2023-11,11745,13659,3034,327,346,14189,1305,270,25786,1788,72449
12,2023,2023-12,10642,9001,3138,216,321,12169,995,20,28358,1525,66385
1,2024,2024-01,9721,8779,3086,187,164,12861,1391,2589,44097,2729,85604
2,2024,2024-02,6247,5002,2578,189,200,10828,2126,80,50574,5723,83547
3,2024,2024-03,8001,6183,3577,237,277,13355,1729,175,39053,7611,80198
4,2024,2024-04,12473,14840,4418,258,313,9945,741,0,13906,3349,60243
5,2024,2024-05,16438,18351,6435,275,616,10264,1009,100,13035,3951,70474
6,2024,2024-06,12417,17250,4497,257,718,12024,1382,0,10340,2978,61863
7,2024,2024-07,13409,14723,4524,237,763,15321,1707,0,15261,2705,68650
8,2024,2024-08,11655,13970,3643,223,870,16233,2212,0,15811,3111,67728
9,2024,2024-09,11734,15768,3988,455,391,18078,1797,25,13692,5460,71388
10,2024,2024-10,14184,18754,5009,295,396,17629,1342,0,11092,2483,71184
11,2024,2024-11,15223,20977,4679,217,1090,14637,1519,0,8619,2215,69176
12,2024,2024-12,13755,20958,4093,381,484,16164,1507,0,9628,2021,68991
1,2025,2025-01,12606,18314,4174,174,776,19354,2771,40,6330,3907,68446
2,2025,2025-02,12849,16080,3437,213,764,19757,1313,0,1705,2990,59108
3,2025,2025-03,16846,15429,3150,256,540,20360,1279,0,0,2748,60608
4,2025,2025-04,16590,15339,4205,234,379,13838,659,0,0,1303,52547


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("ricesupply_2011-2025.csv", index_col=0)
df["Bulan"] = pd.to_datetime(df["Bulan"], format="%Y-%m")
df = df.sort_values("Bulan").reset_index(drop=True)

TARGET_COL = "Karawang"  # research target per brief

print(df.shape)
df.head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print()
print(df[[TARGET_COL, "Total Pasokan"]].describe())

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df["Bulan"], df[TARGET_COL], label=TARGET_COL)
plt.title(f"{TARGET_COL} Rice Supply, 2011\u20132025")
plt.xlabel("Date")
plt.ylabel("Supply (ton)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. Analisis Musiman (Seasonality Analysis)

In [ ]:
df["month"] = df["Bulan"].dt.month
df["year"] = df["Bulan"].dt.year

monthly_avg = df.groupby("month")[TARGET_COL].mean()
month_names = ["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Ags","Sep","Okt","Nov","Des"]

plt.figure(figsize=(10, 4))
bars = plt.bar(month_names, monthly_avg.values, color="#4C72B0")
lowest_idx = monthly_avg.values.argmin()
highest_idx = monthly_avg.values.argmax()
bars[lowest_idx].set_color("#C44E52")
bars[highest_idx].set_color("#55A868")
plt.title(f"Rata-rata Pasokan Bulanan {TARGET_COL} (2011\u20132025)")
plt.ylabel("Rata-rata Pasokan (ton)")
plt.grid(alpha=0.3, axis="y")
plt.show()

lowest_month = month_names[lowest_idx]
highest_month = month_names[highest_idx]
print(f"Bulan dengan pasokan rata-rata terendah: {lowest_month} ({monthly_avg.values[lowest_idx]:,.0f} ton)")
print(f"Bulan dengan pasokan rata-rata tertinggi: {highest_month} ({monthly_avg.values[highest_idx]:,.0f} ton)")

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

series = df.set_index("Bulan")[TARGET_COL]
decomposition = seasonal_decompose(series, model="additive", period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomposition.observed.plot(ax=axes[0], title="Observed")
decomposition.trend.plot(ax=axes[1], title="Trend")
decomposition.seasonal.plot(ax=axes[2], title="Seasonal")
decomposition.resid.plot(ax=axes[3], title="Residual")
plt.tight_layout()
plt.show()

## 6. LSTM Forecasting — Data Preparation

In [ ]:
from sklearn.preprocessing import MinMaxScaler

WINDOW = 12          # 12-month lookback
TEST_MONTHS = 12     # holdout for evaluation

values = df[TARGET_COL].values.reshape(-1, 1).astype("float32")

scaler = MinMaxScaler()
scaled = scaler.fit_transform(values)

def make_sequences(arr, window):
    X, y = [], []
    for i in range(len(arr) - window):
        X.append(arr[i:i + window, 0])
        y.append(arr[i + window, 0])
    return np.array(X), np.array(y)

X_all, y_all = make_sequences(scaled, WINDOW)
X_all = X_all.reshape((X_all.shape[0], X_all.shape[1], 1))

X_train, X_test = X_all[:-TEST_MONTHS], X_all[-TEST_MONTHS:]
y_train, y_test = y_all[:-TEST_MONTHS], y_all[-TEST_MONTHS:]

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## 7. Build & Train the LSTM Model

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)

model = keras.Sequential([
    keras.layers.Input(shape=(WINDOW, 1)),
    keras.layers.LSTM(64, return_sequences=False),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse")
model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="loss", patience=15, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=8,
    callbacks=[early_stop],
    verbose=0,
)

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"])
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE (scaled)")
plt.grid(alpha=0.3)
plt.show()

## 8. Evaluation (RMSE, MAE, MAPE)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

y_pred_scaled = model.predict(X_test, verbose=0)

y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_actual = scaler.inverse_transform(y_pred_scaled).flatten()

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae = mean_absolute_error(y_test_actual, y_pred_actual)
mape = np.mean(np.abs((y_test_actual - y_pred_actual) / y_test_actual)) * 100

print(f"RMSE: {rmse:,.2f} ton")
print(f"MAE:  {mae:,.2f} ton")
print(f"MAPE: {mape:,.2f} %")

test_dates = df["Bulan"].iloc[-TEST_MONTHS:].values
plt.figure(figsize=(12, 5))
plt.plot(test_dates, y_test_actual, label="Actual", marker="o")
plt.plot(test_dates, y_pred_actual, label="Predicted", marker="x")
plt.title(f"{TARGET_COL}: Actual vs Predicted (Holdout, last {TEST_MONTHS} months)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 9. Forecast 12 Bulan ke Depan

Retrain on the full series (train + holdout) before forecasting forward, then recursively predict 12 months ahead.

In [ ]:
# Refit on the entire series for the production forecasting model
final_model = keras.models.clone_model(model)
final_model.compile(optimizer="adam", loss="mse")
final_model.fit(X_all, y_all, epochs=200, batch_size=8, callbacks=[early_stop], verbose=0)

FORECAST_HORIZON = 12
window_buffer = list(scaled[-WINDOW:, 0])
forecast_scaled = []

for _ in range(FORECAST_HORIZON):
    x_input = np.array(window_buffer[-WINDOW:]).reshape((1, WINDOW, 1))
    next_val = final_model.predict(x_input, verbose=0)[0, 0]
    forecast_scaled.append(next_val)
    window_buffer.append(next_val)

forecast_actual = scaler.inverse_transform(np.array(forecast_scaled).reshape(-1, 1)).flatten()

last_date = df["Bulan"].iloc[-1]
forecast_dates = pd.date_range(last_date + pd.offsets.MonthBegin(1), periods=FORECAST_HORIZON, freq="MS")

bulan_id = ["Januari","Februari","Maret","April","Mei","Juni","Juli","Agustus","September","Oktober","November","Desember"]

def fmt_id(x):
    return f"{x:,.0f}".replace(",", ".")

forecast_df = pd.DataFrame({
    "Bulan": forecast_dates,
    "Forecast_ton": forecast_actual,
})
forecast_df["label"] = forecast_df["Bulan"].apply(lambda d: f"{bulan_id[d.month-1]} {d.year}")

print("Forecast 12 Bulan ke Depan:")
for _, row in forecast_df.iterrows():
    print(f"Forecast {row['label']} : {fmt_id(row['Forecast_ton'])} ton")

## 10. Knowledge Extraction (Internal RAG)

The knowledge base is built entirely from the dataset itself — monthly averages, yearly trend, seasonal extremes, descriptive statistics, seasonal insight, and the forecast just produced. No external documents are used (Internal RAG / Data-to-Text RAG).

In [ ]:
knowledge_docs = []
_id = 0

def add_doc(text, category):
    global _id
    knowledge_docs.append({"id": f"doc_{_id}", "text": text, "category": category})
    _id += 1

# Rata-rata bulanan
for m_idx, avg_val in monthly_avg.items():
    add_doc(
        f"Rata-rata pasokan beras {TARGET_COL} pada bulan {bulan_id[m_idx-1]} (periode 2011-2025) "
        f"adalah {fmt_id(avg_val)} ton.",
        "rata_rata_bulanan",
    )

# Tren tahunan
yearly_avg = df.groupby("year")[TARGET_COL].mean()
trend_direction = "meningkat" if yearly_avg.iloc[-1] > yearly_avg.iloc[0] else "menurun"
add_doc(
    f"Rata-rata pasokan tahunan {TARGET_COL} bergerak dari {fmt_id(yearly_avg.iloc[0])} ton "
    f"pada tahun {yearly_avg.index[0]} menjadi {fmt_id(yearly_avg.iloc[-1])} ton pada tahun "
    f"{yearly_avg.index[-1]}, secara umum {trend_direction} dalam jangka panjang.",
    "tren_tahunan",
)

# Bulan tertinggi & terendah
add_doc(
    f"Bulan dengan rata-rata pasokan {TARGET_COL} TERENDAH secara historis adalah {lowest_month} "
    f"dengan rata-rata {fmt_id(monthly_avg.values[lowest_idx])} ton.",
    "ekstrem_musiman",
)
add_doc(
    f"Bulan dengan rata-rata pasokan {TARGET_COL} TERTINGGI secara historis adalah {highest_month} "
    f"dengan rata-rata {fmt_id(monthly_avg.values[highest_idx])} ton.",
    "ekstrem_musiman",
)

# Statistik deskriptif
desc = df[TARGET_COL].describe()
add_doc(
    f"Statistik deskriptif pasokan {TARGET_COL} (2011-2025): rata-rata {fmt_id(desc['mean'])} ton, "
    f"median {fmt_id(desc['50%'])} ton, standar deviasi {fmt_id(desc['std'])} ton, "
    f"minimum {fmt_id(desc['min'])} ton, maksimum {fmt_id(desc['max'])} ton.",
    "statistik_deskriptif",
)

# Insight musiman
add_doc(
    f"Pasokan beras {TARGET_COL} memiliki pola musiman yang konsisten: titik terendah terjadi pada "
    f"periode Januari-Februari, dan puncak pasokan terjadi pada periode April-Mei. Pola ini sejalan "
    f"dengan musim panen di Karawang, di mana hasil panen mulai masuk ke pasar pada bulan April-Mei "
    f"setelah masa tanam pada akhir tahun sebelumnya.",
    "insight_musiman",
)

# Forecast 12 bulan ke depan
for _, row in forecast_df.iterrows():
    add_doc(
        f"Forecast pasokan {TARGET_COL} untuk {row['label']} adalah {fmt_id(row['Forecast_ton'])} ton "
        f"berdasarkan model LSTM.",
        "forecast",
    )

print(f"Total knowledge documents: {len(knowledge_docs)}")
for d in knowledge_docs[:3]:
    print("-", d["text"])

## 11. Embedding (Sentence Transformer)

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_texts = [d["text"] for d in knowledge_docs]
doc_embeddings = embedder.encode(doc_texts, show_progress_bar=True).tolist()

print(f"Encoded {len(doc_embeddings)} documents, embedding dim: {len(doc_embeddings[0])}")

## 12. ChromaDB Vector Store

In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="rice_supply_knowledge")

collection.add(
    ids=[d["id"] for d in knowledge_docs],
    documents=doc_texts,
    embeddings=doc_embeddings,
    metadatas=[{"category": d["category"]} for d in knowledge_docs],
)

print(f"\u2713 ChromaDB collection ready with {collection.count()} documents")

## 13. Retrieval

In [ ]:
def retrieve(query: str, k: int = 4):
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    return results["documents"][0]

# quick test
for doc in retrieve("pola musiman pasokan Karawang", k=3):
    print("-", doc)

## 14. Gemini — Explainable Forecast

In [ ]:
gemini_model = genai.GenerativeModel(GEMINI_MODEL_NAME)

def explain_forecast(forecast_df, query: str, k: int = 5) -> str:
    context_docs = retrieve(query, k=k)
    context_text = "\n".join(f"- {d}" for d in context_docs)
    forecast_text = "\n".join(
        f"Forecast {row['label']} : {fmt_id(row['Forecast_ton'])} ton"
        for _, row in forecast_df.iterrows()
    )

    prompt = f"""Kamu adalah analis supply chain beras. Berikut adalah hasil forecast LSTM untuk pasokan beras Karawang:

{forecast_text}

Berikut adalah pengetahuan internal (knowledge base) yang relevan, diekstrak dari data historis 2011-2025:
{context_text}

Tulis penjelasan (explanation) atas hasil forecast di atas dalam Bahasa Indonesia, dengan format:

Explanation:
<penjelasan pola/tren forecast dikaitkan dengan pola historis>

Risiko:
- <risiko 1>
- <risiko 2>

Rekomendasi:
- <rekomendasi 1>
- <rekomendasi 2>
"""

    response = gemini_model.generate_content(prompt)
    return response.text

explanation = explain_forecast(forecast_df, "pola musiman dan risiko pasokan Karawang")
print(explanation)

## 15. RAGAS Evaluation

Evaluates the RAG pipeline (retrieval + Gemini-generated answers) on faithfulness, answer relevancy, context precision, and context recall, using a small set of questions with manually-written ground truths derived from the EDA above.

RAGAS's API changes fairly often between versions — if `evaluate()` raises a `TypeError` about unexpected keyword arguments, check `ragas.__version__` printed below and adjust the call (e.g. some versions expect `llm=`/`embeddings=` directly, others expect a `run_config`).

In [ ]:
import ragas
print("ragas version:", ragas.__version__)

eval_questions = [
    "Apa pola musiman pasokan beras Karawang?",
    "Bulan apa yang memiliki rata-rata pasokan beras Karawang terendah?",
    "Bulan apa yang memiliki rata-rata pasokan beras Karawang tertinggi?",
]
ground_truths = [
    f"Pasokan terendah terjadi pada bulan {lowest_month}, dan pasokan tertinggi terjadi pada bulan {highest_month}, sejalan dengan musim panen di Karawang.",
    f"{lowest_month}, dengan rata-rata sekitar {fmt_id(monthly_avg.values[lowest_idx])} ton.",
    f"{highest_month}, dengan rata-rata sekitar {fmt_id(monthly_avg.values[highest_idx])} ton.",
]

eval_contexts, eval_answers = [], []
for q in eval_questions:
    ctx = retrieve(q, k=4)
    eval_contexts.append(ctx)
    prompt = "Berdasarkan konteks berikut, jawab pertanyaan secara singkat dan akurat.\n\nKonteks:\n" + \
             "\n".join(f"- {c}" for c in ctx) + f"\n\nPertanyaan: {q}\nJawaban:"
    answer = gemini_model.generate_content(prompt).text.strip()
    eval_answers.append(answer)
    print(f"Q: {q}\nA: {answer}\n")

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

ragas_llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL_NAME, google_api_key=GOOGLE_API_KEY)
ragas_embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GOOGLE_API_KEY)

ragas_dataset = Dataset.from_dict({
    "question": eval_questions,
    "contexts": eval_contexts,
    "answer": eval_answers,
    "ground_truth": ground_truths,
})

try:
    result = evaluate(
        ragas_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    print(result)
except TypeError as e:
    print("\u274c evaluate() call failed, likely an API mismatch with the installed ragas version:", e)
    print("Try removing the llm=/embeddings= kwargs, or check the ragas docs for your installed version.")

## Notes

- **Research framing:** *Explainable Rice Supply Forecasting Using LSTM and Internal Retrieval-Augmented Generation (RAG): A Case Study of Karawang Rice Supply (2011–2025)*. The contribution is using the time-series dataset itself as the knowledge base to explain forecasts, not just as model input — relevant to Explainable AI, Supply Chain Analytics, and Food Supply Chain Forecasting.
- **Rate limits:** Gemini's free tier caps `gemini-2.5-flash` at 5 requests/minute — the explanation and RAGAS cells make several calls, so if you hit a 429, wait ~15s and re-run.
- **Restarting:** Colab runtimes are ephemeral. Re-run all cells from the top after a restart.
- **Extending:** Model B (LSTM + external variables, e.g. weather/rainfall data) and a larger RAGAS question set are natural next steps once this baseline is validated.